# Fase 7: Análise Estatística Rigorosa
O objetivo aqui é validar hipóteses de negócios usando testes estatísticos robustos (não paramétricos, pois variáveis como Nota e Valor Frete raramente seguem distribuição normal).

In [ ]:
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os

# Adiciona o diretório raiz ao path para conseguir importar o config
sys.path.append(os.path.abspath(".."))
from src.config import get_engine

engine = get_engine()

# Buscando a Tabela Fato direto do banco de dados na Nuvem (Neon)
query = "SELECT * FROM analytics.fact_orders"
df = pd.read_sql(query, engine)
print(f"Base carregada: {df.shape[0]} pedidos na tabela fato.")

## Hipótese 1: O Atraso destrói a satisfação do cliente?
Vamos testar estatisticamente se a mediana das notas de pedidos atrasados é MENOR que a de pedidos no prazo.

**Teste Utilizado:** Mann-Whitney U (ideal para notas ordinais de 1 a 5).

In [ ]:
g0 = df.loc[df.flag_atraso == 0, "review_score"].dropna()
g1 = df.loc[df.flag_atraso == 1, "review_score"].dropna()

# H1: A nota de quem recebe NO PRAZO (g0) é MAIOR que quem recebe ATRASADO (g1)
stat, p = stats.mannwhitneyu(g0, g1, alternative="greater")

print(f"Mediana de estrelas (No prazo): {g0.median()}")
print(f"Mediana de estrelas (Atrasado): {g1.median()}")
print(f"P-Valor: {p:.3e}")

if p < 0.05:
    print("\nConclusão: Rejeitamos a Hipótese Nula (H0)! O atraso comprovadamente reduz a nota do cliente de forma ESTATISTICAMENTE SIGNIFICATIVA.")
else:
    print("\nConclusão: Não podemos provar diferença estatística.")

## Hipótese 2: O Custo de Frete varia brutalmente entre regiões?
Vamos testar se há diferença significativa na distribuição de Frete entre os 5 principais estados consumidores.

**Teste Utilizado:** Kruskal-Wallis H-test (alternativa não paramétrica à ANOVA).

In [ ]:
import numpy as np

# Separar as listas de frete dos 5 estados com mais vendas
top_ufs = df["uf_cliente"].value_counts().head(5).index
fretes_por_uf = [df.loc[(df.uf_cliente == uf) & (df.frete_total.notnull()), "frete_total"] for uf in top_ufs]

stat, p = stats.kruskal(*fretes_por_uf)
print(f"Kruskal-Wallis P-Valor: {p:.3e}\n")

if p < 0.05:
    print("Conclusão: O estado influencia significativamente o valor do frete pago pelo cliente.")

plt.figure(figsize=(10,6))
sns.boxplot(data=df[df["uf_cliente"].isin(top_ufs)], x="uf_cliente", y="frete_total", order=top_ufs, palette="viridis")
plt.ylim(0, 100)
plt.title("Distribuição do Frete Pago nos 5 Principais Estados", fontsize=14)
plt.xlabel("Estado (UF)")
plt.ylabel("Valor do Frete (R$)")
plt.show()